## 3.4 CartPoleをQ学習で制御をxArm6でやる

In [18]:
# 必要なパッケージのインストール
%pip install numpy matplotlib pyrealsense2 xarm-python-sdk opencv-python sounddevice JSAnimation

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for JSAnimation: filename=jsanimation-0.1-py3-none-any.whl size=11472 sha256=d82b03f0580f8836ea387665f88344d504b2217b906a9be9aca4bfaa75983b35
  Stored in directory: c:\users\kentayonekura\appdata\local\pip\cache\wheels\2d\6e\2e\c0e2a000cd36fbb86a92b6e7f0042a195ca3a080898a928f9f
Successfully built JSAnimation
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# パッケージのimport
import numpy as np
import matplotlib.pyplot as plt
import pyrealsense2 as rs
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import time
from xarm.wrapper import XArmAPI
import sounddevice as sd

SDK_VERSION: 1.15.3


In [2]:
# 動画の描画関数の宣言
# 参考URL http://nbviewer.jupyter.org/github/patrickmineault
# /xcorr-notebooks/blob/master/Render%20OpenAI%20gym%20as%20GIF.ipynb
from JSAnimation.IPython_display import display_animation
from matplotlib import animation
from IPython.display import display


def display_frames_as_gif(frames):
    """
    Displays a list of frames as a gif, with controls
    """
    plt.figure(figsize=(frames[0].shape[1]/72.0, frames[0].shape[0]/72.0),
               dpi=72)
    patch = plt.imshow(frames[0])
    plt.axis('off')

    def animate(i):
        patch.set_data(frames[i])

    anim = animation.FuncAnimation(plt.gcf(), animate, frames=len(frames),
                                   interval=50)

    anim.save('movie_cartpole.mp4')  # 動画のファイル名と保存です
    display(display_animation(anim, default_mode='loop'))

In [3]:
# 学習の定数の設定
NUM_DIZITIZED = 6  # 各状態の離散値への分割数
GAMMA = 0.99  # 時間割引率
ETA = 0.5  # 学習係数
MAX_STEPS = 200  # 1試行のstep数
NUM_EPISODES = 1000  # 最大試行回数

# ロボットの定数の設定
MAX_TCP_SPEED = 400  # mm/s
MIN_CART_POS = -0.22
MAX_CART_POS = 0.22
MAX_CART_VEL = MAX_TCP_SPEED / 1000.
MAX_POLE_ANGLE = 0.87   # 約50度
MAX_POL_VEL = 2.

In [4]:
class Agent:
    '''CartPoleのエージェントクラスです、棒付き台車そのものになります'''

    def __init__(self, brain):
        self.brain = brain  # エージェントが行動を決定するための頭脳を生成

    def update_Q_function(self, observation, action, reward, observation_next):
        '''Q関数の更新'''
        self.brain.update_Q_table(
            observation, action, reward, observation_next)

    def get_action(self, observation, step):
        '''行動の決定'''
        action = self.brain.decide_action(observation, step)
        return action
    

In [5]:
class Brain:
    '''エージェントが持つ脳となるクラスです、Q学習を実行します'''

    def __init__(self, num_states, num_actions):
        self.num_actions = num_actions  # CartPoleの行動（右に左に押す）の2を取得

        # Qテーブルを作成。行数は状態を分割数^（4変数）にデジタル変換した値、列数は行動数を示す
        self.q_table = np.random.uniform(low=0, high=1, size=(
            NUM_DIZITIZED**num_states, num_actions))


    def bins(self, clip_min, clip_max, num):
        '''観測した状態（連続値）を離散値にデジタル変換する閾値を求める'''
        return np.linspace(clip_min, clip_max, num + 1)[1:-1]

    def digitize_state(self, observation):
        '''観測したobservation状態を、離散値に変換する'''
        cart_pos, cart_v, pole_angle, pole_v = observation
        digitized = [
            np.digitize(cart_pos, bins=self.bins(MIN_CART_POS, MAX_CART_POS, NUM_DIZITIZED)),
            np.digitize(cart_v, bins=self.bins(-MAX_CART_VEL, MAX_CART_VEL, NUM_DIZITIZED)),
            np.digitize(pole_angle, bins=self.bins(-MAX_POLE_ANGLE, MAX_POLE_ANGLE, NUM_DIZITIZED)),
            np.digitize(pole_v, bins=self.bins(-MAX_POL_VEL, MAX_POL_VEL, NUM_DIZITIZED))
        ]
        return sum([x * (NUM_DIZITIZED**i) for i, x in enumerate(digitized)])
    
    def update_Q_table(self, observation, action, reward, observation_next):
        '''QテーブルをQ学習により更新'''
        state = self.digitize_state(observation)  # 状態を離散化
        state_next = self.digitize_state(observation_next)  # 次の状態を離散化
        Max_Q_next = max(self.q_table[state_next][:])
        self.q_table[state, action] = self.q_table[state, action] + \
            ETA * (reward + GAMMA * Max_Q_next - self.q_table[state, action])

    def decide_action(self, observation, episode):
        '''ε-greedy法で徐々に最適行動のみを採用する'''
        state = self.digitize_state(observation)
        epsilon = 0.5 * (1 / (episode + 1))

        if epsilon <= np.random.uniform(0, 1):
            action = np.argmax(self.q_table[state][:])
        else:
            action = np.random.choice(self.num_actions)  # 0,1の行動をランダムに返す
        return action
    

In [9]:
class RobotEnvironment:
    '''CartPoleを模したロボット環境クラスです、xArm6とRealSenseを用いて実装します'''

    def __init__(self):
        self.step_num = 0  # ステップ数
        self.last_time = time.perf_counter()  # フレームレート計測用の変数
        self.last_cart_pos = 0.0  # 台車の前回位置 [m]
        self.last_pole_angle = 0.0  # 棒の前回角度 [rad]
        self.image_bgr = None  # BGR画像

    def reset(self):
        '''CartPole環境のリセット'''
        # 1回チャイム音を鳴らす（人にロボットをリセットさせる合図）
        self._play_down_chime()
        # リセットされるまで待つwait
        time.sleep(1)

        self.step_num = 0  # ステップ数をリセット
        self.arm.set_position(x=360, y=0, z=370, roll=120, pitch=-90, yaw=60, speed=200, is_radian=False, wait=False)
        time.sleep(2)

        # チャイム音を鳴らす（人にロボットが動き始めることを知らせる合図）
        self._play_up_chime()

        self.last_time = time.perf_counter()  # フレームレート計測用の変数
        observation = self.observation()
        return observation
    
    def _init_rs(self):
        '''Realsenseの初期化'''
        # RealSenseパイプラインの設定
        self.pipeline = rs.pipeline()
        config = rs.config()

        # RGBとDepthストリームを有効にする
        config.enable_stream(rs.stream.depth, 424, 240, rs.format.z16, 60)
        config.enable_stream(rs.stream.color, 424, 240, rs.format.bgr8, 60)

        # ストリーミング開始
        self.pipeline.start(config)

        # アライメント設定（DepthをRGBに合わせる）
        align_to = rs.stream.color
        self.align = rs.align(align_to)

        # 1フレーム取得
        frames = self.pipeline.wait_for_frames()
        
        # フレームをアライメント
        aligned_frames = self.align.process(frames)
        
        # RGB画像Depth画像を取得
        depth_frame = aligned_frames.get_depth_frame()

        # カメラの内部パラメータを取得
        self.depth_intrin = depth_frame.profile.as_video_stream_profile().intrinsics

    def _init_xarm(self):
        '''xArm6の初期化'''
        self.arm = XArmAPI("192.168.0.244", is_radian=True)

        if self.arm.warn_code != 0:
            self.arm.clean_warn()
        if self.arm.error_code != 0:
            self.arm.clean_error()
            
        self.arm.motion_enable(enable=True)
        self.arm.set_mode(7) # online cartesian mode
        self.arm.set_state(state=0)

        self.arm.set_position(x=360, y=0, z=370, roll=120, pitch=-90, yaw=60, speed=100, is_radian=False, wait=False)
        time.sleep(3)

    def initialize(self):
        '''CartPole環境の初期化'''
        self._init_rs()
        self._init_xarm()
    
    def wait_observation(self):
        # Realsenseから画像を取得
        # フレーム取得
        return self.pipeline.wait_for_frames()
    
    def observation(self):
        '''CartPole環境の観測'''
        # 返す変数
        cart_pos: float = 0.0   # 台車の位置 [m]
        cart_vel: float = 0.0   # 台車の速度 [m/s]
        pole_angle: float = 0.0 # 棒の角度 [rad]
        pole_vel: float = 0.0   # 棒の角速度 [rad/s]

        # フレーム取得
        frames = self.pipeline.wait_for_frames()
        current_time = current_time = time.perf_counter()
        elapsed_time = current_time - self.last_time
        
        # 速度計算

        # フレームをアライメント
        aligned_frames = self.align.process(frames)
        
        # RGB画像とDepth画像を取得
        color_frame = aligned_frames.get_color_frame()
        depth_frame = aligned_frames.get_depth_frame()

        if not color_frame or not depth_frame:
            return None
        
        # NumPy配列に変換
        color_image = np.asanyarray(color_frame.get_data())
        self.image_bgr = color_image

        # 朱色の点を検出
        hsv_image = cv2.cvtColor(color_image, cv2.COLOR_BGR2HSV)
        lower_orange = np.array([0, 190, 120])
        upper_orange = np.array([15, 255, 180])
        mask_orange = cv2.inRange(hsv_image, lower_orange, upper_orange)

        # 閾値以下のサイズのノイズを除去
        kernel = np.ones((3, 3), np.uint8)
        mask_orange = cv2.morphologyEx(mask_orange, cv2.MORPH_OPEN, kernel)
        mask_orange = cv2.morphologyEx(mask_orange, cv2.MORPH_CLOSE, kernel)
        contours_orange, _ = cv2.findContours(mask_orange, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # 緑色の点を検出
        hsv_image = cv2.cvtColor(color_image, cv2.COLOR_BGR2HSV)
        lower_green = np.array([65, 225, 50])
        upper_green = np.array([80, 255, 110])
        mask_green = cv2.inRange(hsv_image, lower_green, upper_green)

        # 閾値以下のサイズのノイズを除去
        kernel = np.ones((3, 3), np.uint8)
        mask_green = cv2.morphologyEx(mask_green, cv2.MORPH_OPEN, kernel)
        mask_green = cv2.morphologyEx(mask_green, cv2.MORPH_CLOSE, kernel)
        contours_green, _ = cv2.findContours(mask_green, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # 3D座標
        green_point_3d = None
        orange_point_3d = None

        if contours_orange:
            # 最大の輪郭を取得
            largest_contour = max(contours_orange, key=cv2.contourArea)

            # 輪郭の中心を計算
            M = cv2.moments(largest_contour)
            if M["m00"] != 0:
                center_x = int(M["m10"] / M["m00"])
                center_y = int(M["m01"] / M["m00"])
                
                # 深度値を取得
                depth_value = depth_frame.get_distance(center_x, center_y)
                
                if depth_value > 0:
                    # 3D座標に変換
                    depth_intrin = depth_frame.profile.as_video_stream_profile().intrinsics
                    point_3d = rs.rs2_deproject_pixel_to_point(
                        depth_intrin, [center_x, center_y], depth_value
                    )
                    orange_point_3d = point_3d

                    #print(f"朱色の点の3D座標: x={orange_point_3d[0]:.3f}m, y={orange_point_3d[1]:.3f}m, z={orange_point_3d[2]:.3f}m")
                    cart_pos = orange_point_3d[0]  # 台車の位置 [m]
                    if elapsed_time > 0:  # ゼロ除算対策
                        cart_vel = (cart_pos - self.last_cart_pos) / elapsed_time  # 台車の速度 [m/s]
                    else:
                        cart_vel = 0.0
                    self.last_cart_pos = cart_pos  # 台車の前回位置を更新
                else:
                    print("有効な深度値が取得できませんでした")

        if contours_green:
            # 最大の輪郭を取得
            largest_contour = max(contours_green, key=cv2.contourArea)

            # 輪郭の中心を計算
            M = cv2.moments(largest_contour)
            if M["m00"] != 0:
                center_x = int(M["m10"] / M["m00"])
                center_y = int(M["m01"] / M["m00"])
                
                # 深度値を取得
                depth_value = depth_frame.get_distance(center_x, center_y)
                
                if depth_value > 0:
                    # 3D座標に変換
                    depth_intrin = depth_frame.profile.as_video_stream_profile().intrinsics
                    point_3d = rs.rs2_deproject_pixel_to_point(
                        depth_intrin, [center_x, center_y], depth_value
                    )
                    green_point_3d = point_3d

                    #print(f"緑色の点の3D座標: x={green_point_3d[0]:.3f}m, y={green_point_3d[1]:.3f}m, z={green_point_3d[2]:.3f}m")
                else:
                    print("有効な深度値が取得できませんでした")
        
        if orange_point_3d is not None and green_point_3d is not None:
            # 朱色の点と緑色の点の2D座標を取得
            orange_2d = np.array([-orange_point_3d[1], -orange_point_3d[0]])
            green_2d = np.array([-green_point_3d[1], -green_point_3d[0]])
            
            # 2Dベクトルを計算
            vector = green_2d - orange_2d
            
            # 角度を計算（atan2を使用して反時計回りを正とする）
            angle_rad = np.arctan2(vector[1], vector[0])
            angle_deg = np.degrees(angle_rad)
            
            pole_angle = angle_rad  # 棒の角度 [rad]
            if elapsed_time > 0:  # ゼロ除算対策
                pole_vel = (pole_angle - self.last_pole_angle) / elapsed_time  # 棒の角速度 [rad/s]
            else:
                pole_vel = 0.0
            self.last_pole_angle = pole_angle  # 棒の前回角度を更新

        print(f"フレームレート: {1/(current_time - self.last_time):.1f} FPS")
        self.last_time = current_time  # フレームレート計測用の変数を更新
        self.step_num += 1

        observation = np.array([cart_pos, cart_vel, pole_angle, pole_vel])
        return observation
    
    def get_image(self):
        # BGR to RGB変換（matplotlib用）
        color_image_rgb = cv2.cvtColor(self.image_bgr, cv2.COLOR_BGR2RGB)
        return color_image_rgb

    def step(self, action):
        '''CartPole環境の1ステップ実行'''
        # xArm6を動かす
        if action == 0:
            self.arm.set_position(x=360, y=240, z=370, roll=120, pitch=-90, yaw=60, speed=MAX_TCP_SPEED, is_radian=False, wait=False)
        else:
            self.arm.set_position(x=360, y=-240, z=370, roll=120, pitch=-90, yaw=60, speed=MAX_TCP_SPEED, is_radian=False, wait=False)

        # 環境の観測
        observation = self.observation()

        # 終了判定
        done = bool(
            observation[0] < MIN_CART_POS         # [m]
            or observation[0] > MAX_CART_POS      # [m]
            or observation[2] < -MAX_POLE_ANGLE   # 約-50度
            or observation[2] > MAX_POLE_ANGLE    # 約50度
        )
        if self.step_num >= MAX_STEPS:
            done = True

        # 報酬の設定
        if done and bool(
            observation[2] > -0.26      # -15度
            and observation[2] < 0.26   # 15度
            ):
            reward = 1.0
        else:
            reward = 0.0

        info = {}

        return observation, reward, done, info

    def _play_beep(self, frequency=800, duration=0.5, sample_rate=44100):
        """ビープ音を再生"""
        t = np.linspace(0, duration, int(sample_rate * duration), False)
        wave = np.sin(frequency * 2 * np.pi * t)
        sd.play(wave, sample_rate)
        sd.wait()  # 音の再生完了まで待機

    def _play_up_chime(self):
        """チャイム音を再生（複数の音程）"""
        frequencies = [523, 659, 784, 1047]  # C, E, G, C (ドミソド)
        for freq in frequencies:
            self._play_beep(freq, 0.3)

    def _play_down_chime(self):
        """チャイム音を再生（複数の音程）"""
        frequencies = [1047, 784, 659, 523]  # C, E, G, C (ドミソド)
        for freq in frequencies:
            self._play_beep(freq, 0.3)

In [6]:
num_states = 4  # cart_pos, cart_vel, pole_angle, pole_vel の4変数
num_actions = 2  # CartPoleの行動（右に左に押す）の2を取得
brain1 = Brain(num_states, num_actions)

In [7]:
class Environment:
    '''CartPoleを実行する環境のクラスです'''

    def __init__(self):
        self.agent = Agent(brain1)  # 環境内で行動するAgentを生成
        self.robot_env = RobotEnvironment()  # ロボット環境を生成

    def run(self):
        '''実行'''
        complete_episodes = 0  # 195step以上連続で立ち続けた試行数
        is_episode_final = False  # 最終試行フラグ
        last_reward_step = 10 # 最後に報酬を得たステップ数
        clip_step = 200
        frames = []  # 動画用に画像を格納する変数

        self.robot_env.initialize()  # ロボット環境の初期化

        for episode in range(NUM_EPISODES):  # 試行数分繰り返す
            observation = self.robot_env.reset()  # 環境の初期化

            for step in range(MAX_STEPS):  # 1エピソードのループ

                if is_episode_final is True:  # 最終試行ではframesに各時刻の画像を追加していく
                    frames.append(self.robot_env.get_image())

                # 行動を求める
                action = self.agent.get_action(observation, episode)

                # 行動a_tの実行により、s_{t+1}, r_{t+1}を求める
                observation_next, _, done, _ = self.robot_env.step(
                    action)  # rewardとinfoは使わないので_にする

                # 報酬を与える
                if done:  # ステップ数が200経過するか、一定角度以上傾くとdoneはtrueになる
                    if step < last_reward_step:
                        reward = -1  # 途中でこけたら罰則として報酬-1を与える
                        complete_episodes = 0  # last_reward_step以上連続で立ち続けた試行数をリセット
                    else:
                        reward = 1  # 立ったまま終了時は報酬1を与える
                        complete_episodes += 1  # 連続記録を更新
                        if last_reward_step < clip_step:
                            last_reward_step += 10
                else:
                    reward = 0  # 途中の報酬は0

                # step+1の状態observation_nextを用いて,Q関数を更新する
                self.agent.update_Q_function(
                    observation, action, reward, observation_next)

                # 観測の更新
                observation = observation_next

                # 現在の状態の表示
                print(f"Cart position: {observation[0]}, Pole angle: {observation[2]}")

                # 終了時の処理
                if done:
                    print('{0} Episode: Finished after {1} time steps'.format(
                        episode, step + 1))
                    break

            if is_episode_final is True:  # 最終試行では動画を保存と描画
                display_frames_as_gif(frames)
                break

            if complete_episodes >= 10:  # 10連続成功なら
                print('10回連続成功')
                is_episode_final = True  # 次の試行を描画を行う最終試行とする
                

In [ ]:
# main
cartpole_env = Environment()
cartpole_env.run()


ROBOT_IP: 192.168.0.244, VERSION: v2.6.0, PROTOCOL: V1, DETAIL: 6,6,XI1100,XX0000,v2.6.0, TYPE1300: [0, 0]
change protocol identifier to 3
[motion_enable], xArm is not ready to move
[set_state], xArm is ready to move
フレームレート: 29673.6 FPS
フレームレート: 82.0 FPS
Cart position: 0.00023103250714484602, Pole angle: 0.0
フレームレート: 59.2 FPS
Cart position: -0.001563556375913322, Pole angle: 0.0
フレームレート: 58.9 FPS
Cart position: -0.0015663787489756942, Pole angle: 0.0
フレームレート: 59.8 FPS
Cart position: -0.001563556375913322, Pole angle: 0.0
フレームレート: 60.0 FPS
Cart position: 0.00023144952137954533, Pole angle: 0.0
フレームレート: 58.9 FPS
Cart position: -0.0015663787489756942, Pole angle: 0.0
フレームレート: 59.3 FPS
Cart position: 0.00023144952137954533, Pole angle: 0.0
フレームレート: 57.7 FPS
Cart position: -0.0015663787489756942, Pole angle: 0.0
フレームレート: 61.4 FPS
Cart position: -0.0015663787489756942, Pole angle: 0.0
フレームレート: 60.3 FPS
Cart position: 0.0002318665647180751, Pole angle: 0.0
フレームレート: 59.2 FPS
Cart position: 0.

KeyboardInterrupt: 

[SDK][ERROR][2025-10-05 10:18:55][base.py:293] - - [main-socket] recv error: [WinError 10054] 既存の接続はリモート ホストに強制的に切断されました。
[SDK][ERROR][2025-10-05 10:18:55][base.py:247] - - [report-socket] recv error: [WinError 10054] 既存の接続はリモート ホストに強制的に切断されました。
